# Acquirium quick start

Acquirium is a metadata + timeseries platform for water systems. The server holds two things about a plant:

1. **a semantic model** — equipment, piping, sensors and units, described with the ASHRAE 223 / WaTr ontologies
2. **the timeseries** of every measured point

The key idea: **you don't query tag names, you query meaning.** You describe what you want in domain terms — a pump, salt concentration, whatever is upstream of the RO — and acquirium finds the points. Code written this way is not specific to a plant: it runs on any plant that has a model.

To get support reach out to [Mete](mailto:saka@mines.edu)!

## Setup
0. Follow the steps in [deployments/WATERTAP/README.md](../../deployments/WATERTAP/README.md) to install requirements
1. Start the server with a config: `acquirium server --config deployments/WATERTAP/scripts/acquirium.toml`
2. Connect:

In [1]:
from acquirium import Acquirium
acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

########################################################################
########################################################################
########################################################################
## For better display of polars dataframes in Jupyter notebooks
import polars as pl
pl.Config.set_tbl_width_chars(1000)
pl.Config.set_fmt_str_lengths(200)

polars.config.Config

## Find entities by ontology classes
`_class` takes plain text. The server resolves it against the ontology, so you don't need to know any ontology URIs to explore:

In [4]:
q = acq.find_entity(_class="Pump", alias="pump")
q.metadata()

pump
str
"""wbs:P1"""
"""wbs:P2"""
"""wbs:intake"""


In [5]:
q = acq.explore().entity("Pump")
q.metadata()

Pump
str
"""wbs:P1"""
"""wbs:P2"""
"""wbs:intake"""


## A query is a description, data comes last
Queries are built step by step and nothing is fetched until you ask. `.metadata()` tells you *which points* matched; `.dataframe()` then pulls the numbers. Let's ask for every salt concentration in the plant:

In [6]:
q = (acq.find_all_data()
        .filter_by_substance("constituent salt")
        .filter_by_quantity_kind("mass concentration"))
q.metadata().unique()

0
str
"""wbs:storage-tank-3-out-tds-concentration"""
"""wbs:intake-in-tds-concentration"""
"""wbs:PXR-brine-out-tds-concentration"""


In [15]:
q = (acq.explore().measurement().where(substance = "constituent salt", quantity_kind = "mass_concentration"))
q.metadata()

data
str
"""wbs:storage-tank-3-out-tds-concentration"""
"""wbs:intake-in-tds-concentration"""
"""wbs:PXR-brine-out-tds-concentration"""


Three points matched: 
- seawater feed
- brine discharge 
- product water 

We can access to their timeseries values with:

In [22]:
q.dataframe(shape="wide", cast_value="float").drop_nulls().tail(3)

time,data__wbs:intake-in-tds-concentration,data__wbs:PXR-brine-out-tds-concentration,data__wbs:storage-tank-3-out-tds-concentration
"datetime[μs, UTC]",f64,f64,f64
2026-08-04 22:08:10.466349 UTC,36.357657,62.045675,0.236192
2026-08-04 22:08:25.613162 UTC,36.438138,62.224976,0.237601
2026-08-04 22:08:40.204383 UTC,36.402763,62.041978,0.2362


## The topology is queryable
For questions about *where* such as what feeds this unit, what is measured downstream of that one, etc.; We can explore and find the related equipment. For instance, let's find the RO membraine and it's upstream pump:

In [23]:
(acq.find_entity(_class="reverse osmosis membrane", alias="ro")
    .find_related(_class="Pump", alias="feed_pump", direction="upstream", hops=3)
    .metadata())

ro,feed_pump
str,str
"""wbs:RO""","""wbs:P2"""
"""wbs:RO""","""wbs:P1"""


In [24]:
q = acq.explore().entity("reverse osmosis membrane").related("pump",direction="upstream")
q.metadata()

reverse osmosis membrane,pump
str,str
"""wbs:RO""","""wbs:P2"""
"""wbs:RO""","""wbs:P1"""


## Units
Every property can carry a QUDT unit and Acquirium can convert these units to each other:

In [27]:
q = (acq.explore().measurement().where(substance = "constituent salt", quantity_kind = "mass_concentration"))
data = q.data()
print(data.units())
data.convert_to("mg/L").dataframe().drop_nulls().tail(3)

{'data': 'http://qudt.org/vocab/unit/KiloGM-PER-M3'}


time,data__wbs:intake-in-tds-concentration,data__wbs:PXR-brine-out-tds-concentration,data__wbs:storage-tank-3-out-tds-concentration
"datetime[μs, UTC]",f64,f64,f64
2026-08-04 22:09:56.295818 UTC,36398.711388,61614.546584,233.779613
2026-08-04 22:10:10.793112 UTC,36645.763657,62211.095059,238.600054
2026-08-04 22:10:25.853436 UTC,36602.621279,61936.889633,237.208973


## Where to go next
- `watertap-1.ipynb` — the client reference: every feature, plus the internals (query graph, generated SPARQL)
- `regulation.ipynb` — a real application: checking this plant against the California Ocean Plan and Title 22 drinking water standards

